In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import os
import re
import pickle
from dotenv import load_dotenv

import torch
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer, CrossEncoder

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:
query = "Trafikte geçiş üstünlüğü öncelik sıralaması nedir?"
top_k = 20

### Dense Search

In [6]:
client = QdrantClient("localhost", port=6333)

In [7]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
embed_model_name = "intfloat/multilingual-e5-small"
embed_model = SentenceTransformer(embed_model_name, token=hf_token, device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7703.29it/s]


In [8]:
query_emb = embed_model.encode([query])[0]
dense_results = client.query_points(
    collection_name="legal_docs", query=query_emb.tolist(), limit=top_k
).points
dense_results = [(str(r.payload["chunk_id"]), r.score) for r in dense_results]

### BM25 Search

In [9]:
# Clean the texts
def tokenize_tr(text: str) -> list[str]:
    text = text.replace("İ", "i").replace("I", "ı").lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)

    return text.split()

In [10]:
with open("../data/processed/bm25_index.pkl", "rb") as f:
    bm25_index = pickle.load(f)

In [11]:
bm25 = bm25_index["bm25"]
chunks = bm25_index["chunks"]

In [12]:
tokenized_query = tokenize_tr(query)
scores = bm25.get_scores(tokenized_query)
scored = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]

sparse_results = [(chunks[idx]["chunk_id"], float(score)) for idx, score in scored]

### Hybrid Search

In [13]:
rrf_scores = {}

# Dense sonuçlarını ekle
for rank, (chunk_id, _) in enumerate(dense_results):
    rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (60 + rank + 1)

# BM25 sonuçlarını ekle
# BM25 chunk dict döndürüyor, chunk_id'yi metadata'dan al
for rank, (chunk_id, _) in enumerate(sparse_results):
    rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (60 + rank + 1)

# Sırala
sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

# chunk_id → chunk dict mapping oluştur
id_to_chunk = {c["chunk_id"]: c for c in chunks}

In [14]:
hybrid_results = [
    {**id_to_chunk[cid], "rrf_score": rrf_scores[cid]}
    for cid in sorted_ids
    if cid in id_to_chunk
]

In [15]:
[(r["chunk_id"], r["rrf_score"]) for r in hybrid_results[:5]]

[('trafik_kanunu_75', 0.03278688524590164),
 ('trafik_kanunu_60', 0.031024531024531024),
 ('trafik_kanunu_58', 0.030798389007344232),
 ('trafik_kanunu_78', 0.0304147465437788),
 ('trafik_kanunu_88', 0.030330882352941176)]

### ReRanker

In [16]:
reranker_model = CrossEncoder(
    model_name_or_path="BAAI/bge-reranker-base", max_length=512
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6490.58it/s]


In [17]:
pairs = [(query, chunk["text"]) for chunk in chunks]

scores = reranker_model.predict(pairs, show_progress_bar=False)

scored_chunks = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)

reranker_results = [
    {**chunk, "reranker_score": float(score)} for chunk, score in scored_chunks[:5]
]

In [18]:
[(r["chunk_id"], r["reranker_score"]) for r in reranker_results]

[('trafik_kanunu_55', 0.8356351852416992),
 ('trafik_kanunu_75', 0.824079692363739),
 ('trafik_kanunu_59', 0.8202239871025085),
 ('trafik_kanunu_53', 0.7009274959564209),
 ('trafik_kanunu_77', 0.6870260238647461)]

In [22]:
reranker_results

[{'chunk_id': 'trafik_kanunu_55',
  'text': '52 – Sürücüler:\na) Kavşaklara yaklaşırken, dönemeçlere girerken, tepe üstlerine yaklaşırken, dönemeçli\nyollarda ilerlerken, yaya geçitlerine, hemzemin geçitlere, tünellere, dar köprü ve menfezlere\nyaklaşırken, yapım ve onarım alanlarına girerken, hızlarını azaltmak,\nb) Hızlarını, kullandıkları aracın yük ve teknik özelliğine, görüş, yol, hava ve trafik\ndurumunun gerektirdiği şartlara uydurmak,\nc) Diğer bir aracı izlerken yukarıdaki fıkrada belirlenen durumları göz önünde tutarak\ngüvenli bir mesafe bırakmak,\nd) Kol ve grup halinde araç kullananlar, araçları arasında yönetmelikte belirtilen\nesaslara uygun olarak diğer araçların güvenle girebilecekleri açıklıklar bulundurmak,\nZorundadırlar.\n(Değişik: 21/5/1997-4262/4 md.) Bu madde hükmüne uymayan sürücüler 1 800 000\nlira para cezası ile cezalandırılırlar.\nÜÇÜNCÜ BÖLÜM\nSürücülerin Uyacağı Diğer KurallarDönüş kuralları:',
  'metadata': {'madde_no': 55, 'char_count': 877},
  'reranke